# GRPO：PCB placement 品質最佳化 (genpcb)

從 **SFT 暖身後的 adapter** 接續，用 GRPO 直接最佳化 placement reward。

- **reward**：Tier-1 確定性 placement（HPWL+重疊+越界+decap），純 Python、每 rollout 即時算
- **group**：同 prompt(同 netlist) 內 `num_generations` 個 completion 做 group-relative advantage
- **起點**：SFT adapter（先跑 train_sft_colab.ipynb），GRPO 接續訓練它

> ⚠️ **本期最佳化的是 placement proxy，不是真實可佈線性**（routing/DRC reward 需 KiCad，未啟用）。
> reward 是精確確定性指標（非 surrogate），但模型可能鑽 proxy 空子 → 需日後 routing reward 驗證。
> ⚠️ KL 參考為 base 模型（PEFT adapter-disable），非 SFT；beta 小，第一輪可接受。

> 執行前：Runtime → A100；Colab Secrets 設 `HF_TOKEN`、`GH_TOKEN`。

## 1. 確認 GPU

In [ ]:
!nvidia-smi

## 2. 安裝套件
需 TRL 含 `GRPOTrainer`（近期版本）。

In [ ]:
!pip -q install -U \
  "transformers>=5.0" "trl>=0.15" "peft>=0.13" \
  "bitsandbytes>=0.44" "accelerate>=1.0" "datasets>=3.0"

## 3. 設定（可直接改這格）

In [ ]:
MODEL           = "google/gemma-4-12b-it"          # 須與 SFT 同一顆
SFT_ADAPTER_DIR = "/content/drive/MyDrive/genpcb/qlora-gemma4-12b"   # train_sft_colab 的輸出
OUTPUT_DIR      = "/content/drive/MyDrive/genpcb/grpo-gemma4-12b"
N_PROMPTS       = 512        # 程序化 prompt 數（seed 與 SFT 錯開）
NUM_GENERATIONS = 8          # 每 prompt 生幾個 → group 大小
MAX_STEPS       = 200        # 煙霧規模；穩定後再拉長
LR              = 1e-6       # GRPO 遠小於 SFT
BETA            = 0.001      # KL 係數
PARSE_FAIL      = -10.0      # malformed floor（較小幅度，避免 group 標準化壓掉品質訊號）
MAX_PROMPT_LEN  = 1024
MAX_COMPLETION_LEN = 512

## 4. Secrets + 掛 Drive + 取得 repo

In [ ]:
import os, sys
from google.colab import drive, userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
drive.mount("/content/drive")
assert os.path.isdir(SFT_ADAPTER_DIR), f"找不到 SFT adapter：{SFT_ADAPTER_DIR}（先跑 train_sft_colab.ipynb）"

GH = userdata.get("GH_TOKEN")
if os.path.isdir("/content/genpcb"):
    !git -C /content/genpcb pull
else:
    !git clone https://{GH}@github.com/timmytsaa/genpcb.git /content/genpcb
!pip -q install -e /content/genpcb
sys.path.insert(0, "/content/genpcb/src")

## 5. Prompt 集 + reward 函式（皆 import 自 repo）
GRPO 自生 completion，不需標籤；prompt 為 placement 任務（板框+宣告+netlist+PLACE）。

In [ ]:
from datasets import Dataset
from genpcb.train.grpo import build_prompts, make_reward_fn
from genpcb.rewards import DEFAULT_WEIGHTS

prompt_ds = Dataset.from_list(build_prompts(N_PROMPTS, seed0=10000))   # 與 SFT(0..) 錯開
reward_fn = make_reward_fn(DEFAULT_WEIGHTS, parse_fail=PARSE_FAIL)
print(len(prompt_ds), "prompts")
print(prompt_ds[0]["prompt"][:200])

## 6. 載入 base(4-bit) + SFT adapter（接續訓練）
adapter 設 `is_trainable=True`；GRPO 繼續訓練這個 LoRA。KL 參考為 base（adapter 關閉）。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, SFT_ADAPTER_DIR, is_trainable=True)
model.config.use_cache = False

## 7. GRPO 訓練
生成是瓶頸（每 prompt 生 `NUM_GENERATIONS` 個，HF generate，無 vLLM）→ 煙霧規模 `MAX_STEPS` 先設小。

In [ ]:
from trl import GRPOConfig, GRPOTrainer

args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    beta=BETA,
    num_generations=NUM_GENERATIONS,
    per_device_train_batch_size=NUM_GENERATIONS,   # 1 prompt 的 group / device step
    gradient_accumulation_steps=2,
    max_prompt_length=MAX_PROMPT_LEN,
    max_completion_length=MAX_COMPLETION_LEN,
    max_steps=MAX_STEPS,
    temperature=1.0,
    logging_steps=5,
    save_steps=50,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    use_vllm=False,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=args,
    train_dataset=prompt_ds,
    processing_class=tok,
)
trainer.train()

## 8. 存 adapter

In [ ]:
trainer.save_model(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)
print("GRPO adapter 已存到:", OUTPUT_DIR)

## 9. 評估：GRPO 模型 vs 程序化基線（held-out netlist）
對全新 netlist（seed 與訓練全錯開）批次生成，比較 reward。
GRPO 直接最佳化 reward，**有機會超越**它暖身用的程序化基線（這就是想看到的）。

In [ ]:
import numpy as np
from genpcb.data.procedural import FAMILIES, generate_board
from genpcb.data.serialize import board_to_dsl, dsl_to_sft_example
from genpcb.rewards import reward_from_completion

model.config.use_cache = True
N_EVAL = 30
r_gen, r_base, valid = [], [], 0
for i in range(N_EVAL):
    fam = FAMILIES[i % len(FAMILIES)]
    board = generate_board(fam, seed=20000 + i)         # held-out
    ex = dsl_to_sft_example(board_to_dsl(board))
    inp = tok(ex["prompt"], return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=MAX_COMPLETION_LEN, do_sample=False)
    gen = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    rg = reward_from_completion(ex["prompt"], gen)
    rb = reward_from_completion(ex["prompt"], ex["completion"])   # 程序化基線
    r_gen.append(rg); r_base.append(rb); valid += (rg > -100)

r_gen, r_base = np.array(r_gen), np.array(r_base)
print(f"合法率（會 parse 且完整）: {valid}/{N_EVAL} = {valid/N_EVAL:.0%}")
print(f"GRPO 模型 reward : mean {r_gen.mean():.3f}  median {np.median(r_gen):.3f}")
print(f"程序化基線 reward : mean {r_base.mean():.3f}")
print(f"勝過基線比例: {(r_gen > r_base).mean():.0%}")

## 調參小抄 / 下一步

- **OOM**：降 `NUM_GENERATIONS`(8→4)、降 `MAX_COMPLETION_LEN`、`gradient_accumulation_steps=1`
- **reward 不升**：GRPO lr 太小/太大都會卡 → 試 5e-7 ~ 5e-6；或 group 太小(調大 `NUM_GENERATIONS`)
- **大量 -10（malformed）**：SFT 暖身不足 → 回頭加 SFT epoch/資料
- **品質崩壞 / 鑽空子**：reward 升但擺位變怪 → 即 proxy 被攻破，需 routing reward（KiCad）把關
- **太慢**：生成是瓶頸；穩定後改 `use_vllm=True`（需相容環境）

**成功標準**：訓練中 reward 平均上升；評估 cell 合法率高、GRPO mean reward ≥ 程序化基線。
下一步 → routing/DRC reward（需 KiCad 環境）併入 reward_fn，才測得到真實可佈線性。